# Cierre: síntesis y tarjeta de cada modelo

## Qué responde este notebook

Resume lo que respondió cada etapa, con el notebook donde está el detalle, y arma una tarjeta por
modelo a partir de la metadata que guardó
[`6.1_modelo_final_y_temporada_actual.ipynb`](6.1_modelo_final_y_temporada_actual.ipynb). Las cifras
de la tarjeta se leen de esos archivos, no se copian a mano. El índice completo de notebooks está en
[`README.md`](README.md).

## 1. Datos y variables (etapas 1 a 3)

- **Fuente**: `nflreadpy`, auditada por fuente en
  [`1.1_calidad_fuentes.ipynb`](1.1_calidad_fuentes.ipynb) y cargada con `weekly/wr/src/data.py`
  ([`1.2_construccion_datos.ipynb`](1.2_construccion_datos.ipynb)). Las alineaciones de 2025 en
  adelante usan otro esquema; se unificaron con las históricas en
  [`1.3_unificacion_alineaciones.ipynb`](1.3_unificacion_alineaciones.ipynb). Las históricas,
  además, usaban códigos de equipo de su época (`OAK`, `SD`); se traducen a los actuales en `data.py`
  (ver [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)).
- **Sin fuga**: cada promedio de un receptor usa solo partidos anteriores
  ([`1.4_promedios_sin_fuga.ipynb`](1.4_promedios_sin_fuga.ipynb)).
- **Lo que mostró el análisis exploratorio** (2.1 a 2.6): recepciones y yardas dependen sobre todo de
  la participación del receptor en su ofensiva; los touchdowns son un evento raro, con la mayoría de
  los partidos en cero.
- **Variables** ([`3.1_ingenieria_features.ipynb`](3.1_ingenieria_features.ipynb) y
  [`3.3_seleccion_estable.ipynb`](3.3_seleccion_estable.ipynb)): de 110 candidatas, 25 para los
  modelos de árboles y 24 para los lineales, elegidas con un orden estable (eliminación recursiva en
  10 submuestras) y el conjunto más chico que no es más de 1% peor que el mejor en validación
  ([`docs/decisions/0005-seleccion-de-variables.md`](../../../docs/decisions/0005-seleccion-de-variables.md)).
  Entre ellas está el total implícito de las líneas de apuestas. Se describen en el
  [glosario](../../../docs/data/GLOSARIO_VARIABLES.md).

## 2. Cómo se mide y cómo se elige

Los modelos predicen valores esperados, que se suman para armar puntos de fantasy. Por eso la métrica
que decide tiene que premiar la media
([`4.1_preparacion_y_metricas.ipynb`](4.1_preparacion_y_metricas.ipynb),
[`docs/decisions/0004-metricas-de-seleccion.md`](../../../docs/decisions/0004-metricas-de-seleccion.md)):

| Resultado | Métrica que decide (validación 2022-2023) | Requisitos para ser elegible |
|---|---|---|
| `receptions`, `receiving_yards` | RMSE | R² fuera de muestra > 0 en validación y en cada año; sesgo adicional ≤ 5%; pendiente de calibración 0.9-1.1 |
| `receiving_tds` | Deviance de Poisson | Los mismos, más D² fuera de muestra > 0 |

El MAE premia la mediana, y en touchdowns la mediana es cero. La prueba (2024-2025) solo se reporta,
y una diferencia entre dos modelos cuenta solo si su intervalo bootstrap por semanas no incluye cero,
ajustado por Bonferroni cuando se comparan varias alternativas a la vez. Pasar esa prueba es
condición necesaria para adoptar un cambio, no suficiente: una mejora de milésimas que obliga a
sumar modelos al flujo semanal no se adopta.

## 3. El modelo elegido de cada resultado

- **Recepciones y yardas**: XGBoost con error cuadrático
  ([`4.2_modelos_recepciones_yardas.ipynb`](4.2_modelos_recepciones_yardas.ipynb)). En los dos,
  RandomForest quedó apenas abajo en RMSE de validación, con intervalos que incluyen cero; por la
  regla de empates se queda XGBoost.
- **Touchdowns**: XGBoost con objetivo Poisson
  ([`4.3_modelos_touchdowns.ipynb`](4.3_modelos_touchdowns.ipynb)). El modelo anterior (Tweedie con
  hiperparámetros por defecto, elegido por MAE) estaba descalibrado y quedaba peor que el promedio
  histórico en deviance; el elegido tiene R² fuera de muestra de 0.09 en validación.
- **Ajuste de hiperparámetros**
  ([`4.4_ajuste_hiperparametros.ipynb`](4.4_ajuste_hiperparametros.ipynb)): una búsqueda compacta no
  mejora ninguno de los tres de forma real con el nivel ajustado por las 30 combinaciones probadas,
  así que se queda la configuración de partida: profundidad 3 en los tres; 200 árboles con tasa 0.03
  en recepciones y yardas, y 400 árboles con tasa 0.03 en touchdowns.

## 4. Alternativas probadas que no se adoptan

Cada alternativa se comparó contra el modelo elegido con la misma regla. Ninguna se adopta:

| Notebook | Alternativa | Resultado |
|---|---|---|
| [`4.5`](4.5_regresion_cuantiles.ipynb) | Rango P10/P50/P90 con regresión de cuantiles | Útil como rango en recepciones y yardas (cobertura de prueba de 83% a 86%); el P50 es la mediana: gana en MAE y pierde en RMSE y R² |
| [`4.6`](4.6_ensamble_simple.ipynb) | Promedios de modelos; mezcla con el modelo de dos etapas en touchdowns | Empates, salvo el ensamble de cuatro modelos en recepciones: mejora real pero de 0.26% del RMSE, que no justifica aplicar cuatro modelos cada semana; en touchdowns ningún peso de mezcla mejora |
| [`4.7`](4.7_jerarquico_touchdowns.ipynb) | Jerárquico con Binomial Negativa en touchdowns | Peor (+0.0102 de deviance) y descalibrado (pendiente 0.85); mezclado con el XGBoost (90/10), empate |
| [`4.8`](4.8_jerarquico_recepciones_yardas.ipynb) | Jerárquico en recepciones y yardas | Peor en los dos (+0.107 y +3.52 de RMSE); en yardas, la escala logarítmica predice cerca de la mediana |
| [`4.9`](4.9_ventana_entrenamiento.ipynb) | Ventanas más cortas, más peso a lo reciente, tendencia por jugador | Empates o peor, ya con el intervalo de 95% |

De estas pruebas, `weekly/wr/models/exploratorio/` guarda solo los modelos de cuantiles de 4.5 y el
jerárquico de 4.7, que reutiliza 4.9 y no se versiona, más la metadata de 4.6 y 4.7. El flujo semanal
no usa ninguno.

## 5. Evaluación

- **Estable año con año** ([`5.1_estabilidad_temporal.ipynb`](5.1_estabilidad_temporal.ipynb)): en
  los cinco cortes de 2021 a 2025, R² fuera de muestra de 0.43-0.47 en recepciones, 0.36-0.39 en
  yardas y 0.0835-0.0973 en touchdowns; los tres cumplen los requisitos en cada año.
- **Mejor que el flujo heredado** ([`5.2_benchmark_vs_legado.ipynb`](5.2_benchmark_vs_legado.ipynb)):
  sobre las mismas semanas 7-18 de 2025, mejor en yardas y touchdowns con diferencias que no incluyen
  cero aun ajustando por las tres comparaciones; en recepciones, mejor en la estimación puntual pero
  empate con el intervalo ajustado. Mejor en MAE en los tres, la métrica del flujo heredado.
- **Explicable** ([`5.3_explicabilidad_y_casos.ipynb`](5.3_explicabilidad_y_casos.ipynb)): el
  orden estable de 3.3, la importancia nativa y SHAP coinciden en la participación reciente y la
  producción de carrera; `depth_team` pesa en SHAP y en el orden estable. El modelo acierta en
  los partidos típicos de cada perfil y se queda corto en las semanas atípicas; por cuartil de
  participación previa queda a menos de 2 yardas de lo real.
- **Temporada en curso**
  ([`6.1_modelo_final_y_temporada_actual.ipynb`](6.1_modelo_final_y_temporada_actual.ipynb)): en las
  semanas 1-4 de 2026, que no se usaron en ningún paso, los tres quedan al nivel de validación en R²
  fuera de muestra. En la métrica principal, yardas y touchdowns quedan un poco por encima de
  validación y prueba, y el modelo subestima (−3.1 yardas por receptor); la pendiente de yardas queda apenas por encima del
  rango (1.105).

## 6. Tarjeta de cada modelo

Leída de `weekly/wr/models/{target}_metadata.json`, la misma metadata que usa el flujo semanal.

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

TARGETS = ("receptions", "receiving_yards", "receiving_tds")
metadata = {t: json.loads(Path(f"../models/{t}_metadata.json").read_text()) for t in TARGETS}

tarjeta = {}
for target, m in metadata.items():
    principal = m["metrica_principal"]
    referencia = m["metricas_referencia"]
    h = m["hiperparametros"]
    tarjeta[target] = {
        "modelo": f"XGBoost {h.get('objective')}, profundidad {h.get('max_depth')}, {h.get('n_estimators')} árboles, tasa {h.get('learning_rate')}",
        "entrenado con": m["entrenado_con"].split(";")[0],
        "variables": len(m["features_arbol_en_orden"]),
        "media de entrenamiento": round(m["media_entrenamiento"], 4),
        "métrica principal": principal,
        "validación 2022-2023": referencia["validacion"][principal],
        "prueba 2024-2025": referencia["prueba"][principal],
        f"2026 (semanas {referencia['temporada_2026']['semanas']})": referencia["temporada_2026"][principal],
        "R² fuera de muestra: validación / prueba / 2026": " / ".join(
            f"{referencia[p]['r2_oos']:.3f}" for p in ("validacion", "prueba", "temporada_2026")),
        "R² fuera de muestra mínimo por año (2021-2025)": min(a["r2_oos"] for a in referencia["walk_forward"].values()),
        "pendiente de calibración: validación / 2026": " / ".join(
            f"{referencia[p]['pendiente_calibracion']:.3f}" for p in ("validacion", "temporada_2026")),
        "cumple requisitos": referencia["cumple_requisitos"],
    }
pd.DataFrame(tarjeta)

,receptions,receiving_yards,receiving_tds
modelo,"XGBoost reg:squarederror, profundidad 3, 200 árboles, tasa 0.03","XGBoost reg:squarederror, profundidad 3, 200 árboles, tasa 0.03","XGBoost count:poisson, profundidad 3, 400 árboles, tasa 0.03"
entrenado con,2016-2025 (temporadas completas),2016-2025 (temporadas completas),2016-2025 (temporadas completas)
variables,25,25,25
media de entrenamiento,2.6981,34.2784,0.2088
métrica principal,rmse,rmse,deviance_poisson
validación 2022-2023,1.9315,29.5002,0.6128
prueba 2024-2025,1.8355,27.9349,0.6221
2026 (semanas 1-4),1.8758,29.5959,0.6288
R² fuera de muestra: validación / prueba / 2026,0.436 / 0.453 / 0.453,0.365 / 0.374 / 0.364,0.090 / 0.091 / 0.093
R² fuera de muestra mínimo por año (2021-2025),0.4292,0.3555,0.0835


In [2]:
for target, m in metadata.items():
    print(f"{target}, limitaciones conocidas:")
    for limitacion in m["limitaciones_conocidas"]:
        print(f"  - {limitacion}")
print("\nrequisitos:", metadata["receptions"]["requisitos"])
print("\nreferencias de la politica de metricas:")
for referencia in metadata["receptions"]["referencias"]:
    print(f"  - {referencia}")

receptions, limitaciones conocidas:
  - predice valores esperados: queda por debajo en semanas de produccion extrema y por encima en semanas sin produccion; no anticipa semanas atipicas (ver 5.3_explicabilidad_y_casos.ipynb)
  - depth_team tiene dato en 95.4% de las filas de 2026 ya jugadas; los faltantes son receptores que no aparecen en la alineacion de esa semana (ver 1.3_unificacion_alineaciones.ipynb)
receiving_yards, limitaciones conocidas:
  - predice valores esperados: queda por debajo en semanas de produccion extrema y por encima en semanas sin produccion; no anticipa semanas atipicas (ver 5.3_explicabilidad_y_casos.ipynb)
  - depth_team tiene dato en 95.4% de las filas de 2026 ya jugadas; los faltantes son receptores que no aparecen en la alineacion de esa semana (ver 1.3_unificacion_alineaciones.ipynb)
receiving_tds, limitaciones conocidas:
  - predice valores esperados: queda por debajo en semanas de produccion extrema y por encima en semanas sin produccion; no anticipa sem

## 7. Limitaciones y pendientes

- **Selección de variables**: se rehízo en `3.3_seleccion_estable.ipynb` sin usar la prueba para
  decidir, pero su regla para elegir el tamaño del conjunto se corrigió después de reportar la prueba
  (ver [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)). La verificación sin ningún uso previo es la
  temporada 2026.
- **Touchdowns**: el R² alcanzable es bajo porque la mayor parte de la variación entre partidos es
  azar; el modelo supera al promedio histórico, pero por poco.
- **A vigilar en el flujo semanal**: la subestimación de las primeras semanas de 2026 y la pendiente
  de calibración: la de touchdowns es la que más varía entre años (de 0.994 a 1.134, fuera del rango en
  2024) y la de yardas en 2026 queda apenas por encima (1.105). El seguimiento semanal las compara
  cada semana contra límites de control
  ([`6.3_seguimiento_semanal.ipynb`](6.3_seguimiento_semanal.ipynb)).
- **Fuera del flujo semanal**: el rango P10/P50/P90
  ([`4.5_regresion_cuantiles.ipynb`](4.5_regresion_cuantiles.ipynb)) y los modelos exploratorios de
  `weekly/wr/models/exploratorio/`.
- **Seguimiento semanal**: qué guarda cada corrida, qué chequeos detienen el flujo, cuándo suena
  una alerta y cuándo se reentrena, en [`6.3_seguimiento_semanal.ipynb`](6.3_seguimiento_semanal.ipynb)
  ([`docs/decisions/0006-seguimiento-semanal.md`](../../../docs/decisions/0006-seguimiento-semanal.md)).
- **Otras posiciones**: cómo adaptar este flujo a QB, RB y TE, en
  [`weekly/GUIA_NUEVA_POSICION.md`](../../GUIA_NUEVA_POSICION.md).

Anterior: [`6.1_modelo_final_y_temporada_actual.ipynb`](6.1_modelo_final_y_temporada_actual.ipynb) ·
Siguiente: [`6.3_seguimiento_semanal.ipynb`](6.3_seguimiento_semanal.ipynb)